# Unified Zeek TSV Preprocessing Notebook

This notebook standardizes preprocessing for Zeek-generated TSV files.

It:
- loads a Zeek TSV file
- normalizes labels
- computes shared engineered features
- computes attack-specific features for DoS HTTP Flood and PortScan
- computes a valid TCP handshake feature
- returns a pandas DataFrame with a consistent feature set

The final output is `processed_df`.


In [3]:
from __future__ import annotations

import numpy as np
import pandas as pd


In [4]:
def normalize_label_name(label: str) -> str:
    label = str(label).strip()
    return (
        label.replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
        .upper()
    )


def compute_and_add_time_elapsed(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["id.orig_h", "id.resp_h", "ts"]).reset_index(drop=True)
    df["time_elapsed"] = (
        df.groupby(["id.orig_h", "id.resp_h"])["ts"]
        .diff()
        .fillna(999999.0)
    )
    return df


def compute_valid_tcp_handshake_feature(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    proto = df["proto"].astype(str).str.lower()
    history = df["history"].astype(str)
    conn_state = df["conn_state"].astype(str).str.upper()

    proto_ok = proto.eq("tcp")
    history_ok = history.str.contains(r"S.*h.*A", regex=True, na=False)
    state_ok = conn_state.isin(["S1", "SF"])

    df["valid_tcp_handshake_feature"] = (proto_ok & history_ok & state_ok).astype(float)
    return df


def add_shared_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    numeric_cols = [
        "ts",
        "duration",
        "orig_bytes",
        "resp_bytes",
        "missed_bytes",
        "orig_pkts",
        "orig_ip_bytes",
        "resp_pkts",
        "resp_ip_bytes",
        "id.resp_p",
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    existing_numeric = [c for c in numeric_cols if c in df.columns]
    if existing_numeric:
        df[existing_numeric] = df[existing_numeric].replace([np.inf, -np.inf], np.nan)
        df[existing_numeric] = df[existing_numeric].fillna(0.0)

    df = compute_and_add_time_elapsed(df)
    df = compute_valid_tcp_handshake_feature(df)

    duration_safe = df["duration"].copy()

    duration_safe = duration_safe.replace([np.inf, -np.inf], np.nan)
    duration_safe = duration_safe.fillna(0.0)
    duration_safe = duration_safe.mask(duration_safe <= 0, 1e-6)

    df["orig_pkt_rate"] = df["orig_pkts"] / duration_safe

    df["orig_byte_rate"] = df["orig_bytes"] / duration_safe
    df["flood_rate"] = df["orig_bytes"] / duration_safe


    df["pkt_asymmetry"] = df["orig_pkts"] / (df["resp_pkts"] + 1.0)


    df["byte_asymmetry"] = df["orig_bytes"] / (df["resp_bytes"] + 1.0)

    engineered_cols = [
        "orig_pkt_rate",
        "orig_byte_rate",
        "pkt_asymmetry",
        "byte_asymmetry",
        "time_elapsed",
        "flood_rate",
        "valid_tcp_handshake_feature",
    ]

    for col in engineered_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            df[col] = df[col].fillna(0.0)

    return df


def add_portscan_attack_features(df: pd.DataFrame, window_seconds: float = 5.0) -> pd.DataFrame:
    df = df.copy()

    required = {"ts", "id.orig_h", "id.resp_p", "orig_pkts", "duration", "conn_state"}
    if not required.issubset(df.columns):
        for col in ["uniq_dst_ports", "pkts_per_port", "scan_duration", "fail_ratio"]:
            df[col] = 0.0
        return df

    df["window_id"] = (df["ts"] // window_seconds).astype(int)
    group_cols = ["id.orig_h", "window_id"]

    agg = df.groupby(group_cols).agg(
        uniq_dst_ports=("id.resp_p", "nunique"),
        total_orig_pkts=("orig_pkts", "sum"),
        scan_duration=("duration", "max"),
        total_flows=("id.orig_h", "size"),
    ).reset_index()

    FAILED_STATES = {"S0", "REJ", "RSTO", "RSTR", "RSTOS0", "RSTRH", "SH", "SHR"}
    df["is_failed_conn"] = df["conn_state"].astype(str).isin(FAILED_STATES).astype(int)

    fail_agg = df.groupby(group_cols).agg(
        failed_flows=("is_failed_conn", "sum"),
    ).reset_index()

    agg = agg.merge(fail_agg, on=group_cols, how="left")
    agg["failed_flows"] = agg["failed_flows"].fillna(0.0)

    agg["pkts_per_port"] = agg["total_orig_pkts"] / agg["uniq_dst_ports"]
    agg["fail_ratio"] = agg["failed_flows"] / agg["total_flows"]

    df = df.merge(
        agg[
            [
                "id.orig_h",
                "window_id",
                "uniq_dst_ports",
                "pkts_per_port",
                "scan_duration",
                "fail_ratio",
            ]
        ],
        on=["id.orig_h", "window_id"],
        how="left",
    )

    for col in ["uniq_dst_ports", "pkts_per_port", "scan_duration", "fail_ratio"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return df

def add_udp_flood_attack_features(
    df: pd.DataFrame,
    window_seconds: float = 5.0,
) -> pd.DataFrame:
    df = df.copy()

    df["is_udp"] = df["proto"].astype(str).str.lower().eq("udp").astype(int)
    df["window_id"] = (df["ts"] // window_seconds).astype(int)

    group_cols = ["id.resp_h", "window_id"]
    udp_df = df[df["is_udp"] == 1]

    agg = udp_df.groupby(group_cols).agg(
        udp_conn_count=("id.orig_h", "size"),
        udp_packets=("orig_pkts", "sum"),
        unique_src_ips=("id.orig_h", "nunique"),
    ).reset_index()

    # Use window duration instead of flow duration
    agg["udp_rate"] = agg["udp_packets"] / window_seconds

    df = df.merge(
        agg,
        on=["id.resp_h", "window_id"],
        how="left",
    )

    for col in [
        "udp_conn_count",
        "udp_packets",
        "udp_rate",
        "unique_src_ips",
    ]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    return df


In [5]:
def load_and_preprocess_data(datapath: str, target_labels: list[str] , window_seconds: float = 5.0) -> pd.DataFrame:
    print(f"Loading data from: {datapath}")
    df = pd.read_csv(datapath, on_bad_lines="skip", delimiter="\t")
    print("Raw shape:", df.shape)
    df.columns = df.columns.str.strip()

    df.drop_duplicates(inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df["label"] = df["label"].astype(str).map(normalize_label_name)
    target_labels = [normalize_label_name(x) for x in target_labels]
    df = df[df["label"].isin(target_labels)].copy()

    df = add_shared_engineered_features(df)
    df = add_portscan_attack_features(df, window_seconds=window_seconds)
    df = add_udp_flood_attack_features(df, window_seconds=window_seconds)

    return df

In [17]:
WINDOW_SECONDS = 5.0
TARGET_LABELS = ["BENIGN", "DOS_HTTP_FLOOD", "PORTSCAN", "DDOS_UDP_FLOOD"]

def load_cicids2017_data() -> pd.DataFrame:
    print("Loading CICIDS2017 data...")
    df_cicids2017_wednesday = load_and_preprocess_data(
        datapath="../data/CICIDS2017/wednesday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    df_cicids2017_friday = load_and_preprocess_data(
        datapath="../data/CICIDS2017/friday_labeled.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    res = pd.concat([df_cicids2017_wednesday, df_cicids2017_friday], ignore_index=True)
    print("Processed shape:", res.shape)
    if "label" in res.columns:
        print(res["label"].value_counts())
    return res

def load_ciciot2023_data() -> pd.DataFrame:
    print("Loading CICIoT2023 data...")
    res = load_and_preprocess_data(
        datapath="../data/CICIoT2023/ciciot2023_labeled_conn.tsv",
        target_labels=TARGET_LABELS,
        window_seconds=WINDOW_SECONDS
    )
    print("Processed shape:", res.shape)
    if "label" in res.columns:
        print(res["label"].value_counts())
    return res

In [13]:
df_cicids2017 = load_cicids2017_data()

Loading CICIDS2017 data...
Loading data from: ../data/CICIDS2017/wednesday_labeled.tsv
Raw shape: (509362, 23)
Loading data from: ../data/CICIDS2017/friday_labeled.tsv
Raw shape: (547353, 23)
Processed shape: (933833, 41)
label
BENIGN            625030
DOS_HTTP_FLOOD    154769
PORTSCAN          154034
Name: count, dtype: int64


In [18]:
df_ciciot2023 = load_ciciot2023_data()

Loading CICIoT2023 data...
Loading data from: ../data/CICIoT2023/ciciot2023_labeled_conn.tsv


C:\Users\Rasmus\AppData\Local\Temp\ipykernel_17936\2570496715.py:3: DtypeWarning: Columns (0: duration, 1: orig_bytes, 2: resp_bytes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(datapath, on_bad_lines="skip", delimiter="\t")


Raw shape: (2609137, 23)
Processed shape: (2192354, 41)
label
DOS_HTTP_FLOOD    1508589
BENIGN             342255
PORTSCAN           216533
DDOS_UDP_FLOOD     124977
Name: count, dtype: int64


In [15]:
df_cicids2017.to_csv(f"cicids2017_preprocessed.tsv", sep='\t', index=False, header=True)

In [19]:
df_ciciot2023.to_csv(f"ciciot2023_preprocessed.tsv", sep='\t', index=False, header=True)

# Validation of data

In [11]:
df_cicids2017.head()

,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,is_failed_conn,uniq_dst_ports,pkts_per_port,scan_duration,fail_ratio,is_udp,udp_conn_count,udp_packets,unique_src_ips,udp_rate
0,1.499285e+09,CXS7062RNvA6jP95x4,0.0.0.0,0,224.0.0.22,0,unknown_transport,-,0.000000,0.0,...,0,1,1.0,0.000000,0.000000,0,0.0,0.0,0.0,0.0
1,1.499276e+09,CLqsJ42jJ2vQJxEMR4,148.251.179.14,3,192.168.10.17,10,icmp,-,0.000000,0.0,...,0,1,1.0,0.000000,0.000000,0,0.0,0.0,0.0,0.0
2,1.499283e+09,CQq6v84asNTMBoWQuh,159.203.79.231,3,192.168.10.16,3,icmp,-,15.115093,1312.0,...,0,1,11.0,15.115093,0.000000,0,0.0,0.0,0.0,0.0
3,1.499256e+09,CFIaxo2raGKXIQTAba,172.16.0.1,53058,192.168.10.50,80,tcp,-,5.002050,0.0,...,0,1,4.0,5.002050,0.000000,0,0.0,0.0,0.0,0.0
4,1.499260e+09,Cm5Sro2xkWqqi3OpXf,172.16.0.1,33296,192.168.10.50,80,tcp,http,12.834921,231.0,...,0,1,92.0,15.836493,0.454545,0,0.0,0.0,0.0,0.0
